# AI model comparison — HTML validation report

Compare two text models on an Istari Digital system branch with the Anthropic Messages API. The model returns structured JSON (matches, conflicts, missing, recommendation); the notebook fills [`report_template.html`](report_template.html) with a source-of-truth trace, then tracks the HTML on the system and commits.

You will:

1. Connect with `istari_labs_helpers` and open a system by **name** and **branch**
2. Resolve the latest AI diff report on the branch (if any) and the two models it compared
3. Re-run only when at least one of those models has a newer revision — or pick two revisions manually
4. Set system prompt and user focus, call Claude for JSON findings, render the HTML report
5. Create a **new report model** on first run, or a **new revision** of the existing report
6. Link the report to the compared resources with a revision relationship

Companion to [CAD parameter validation](validation.ipynb).

### Prerequisites

From the cookbook repository root:

```bash
uv sync --group dev --group ai
uv run python -m ipykernel install --user --name istari-client-cookbook-ai --display-name "Python (istari-client-cookbook + ai)"
```

That installs the Istari Digital client, `anthropic`, and `istari_labs_helpers`, then registers the Jupyter kernel. Reload the window (or reopen the kernel picker) and select **Python (istari-client-cookbook + ai)**.

| Group | Packages | Used for |
|---|---|---|
| **`dev`** | `istari-digital-client`, `python-dotenv`, notebook tooling | Connect, systems, models |
| **`ai`** | `anthropic`, `istari-labs-helpers`, `pdfplumber`, `openpyxl`, `python-docx` | Messages API, helpers, PDF/DOCX/XLSX text extraction |

Other recipes need only `uv sync --group dev`. This notebook also needs **`--group ai`**.

- Credentials in [`samples/.env`](../.env):
  - `ISTARI_REGISTRY_URL`, `ISTARI_PERSONAL_ACCESS_TOKEN`
  - `ANTHROPIC_API_KEY` (or `CLAUDE_API_KEY`) — [Anthropic API keys](https://console.anthropic.com/settings/keys)
- A system with at least two documents on the chosen branch (PDF, DOCX, XLSX, or plain text)
- **Branching** enabled when you commit (Istari Digital web app → **Application Settings** → **Experimental Features**)

### Running order

Run cells top to bottom. §3 prints the outcome and sets `PROCEED`; later cells no-op when there is nothing to compare. If asked, paste revision ids into §4.


## 1 · Connect

Load credentials from [`samples/.env`](../.env). Assert that `ANTHROPIC_API_KEY` is set.


In [ ]:
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, display
from istari_labs_helpers import IstariPlatform

NOTEBOOK_DIR = Path.cwd()
_env = NOTEBOOK_DIR.parent / ".env" if (NOTEBOOK_DIR.parent / ".env").exists() else NOTEBOOK_DIR / "samples" / ".env"
load_dotenv(_env)

# Accept either key name (cookbook uses ANTHROPIC_*; smart-diff uses CLAUDE_*).
if not os.environ.get("ANTHROPIC_API_KEY") and os.environ.get("CLAUDE_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = os.environ["CLAUDE_API_KEY"]

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "Set ANTHROPIC_API_KEY (or CLAUDE_API_KEY) in samples/.env."
)

platform = IstariPlatform.from_env(str(_env))
print(platform)
print(f"Signed in as: {platform.whoami()}")


## 2 · Prep

Set **`SYSTEM_NAME`** (exact system title in the Istari Digital web app) and **`BRANCH_NAME`** (snapshot tag — for example `baseline` or `main`).


In [ ]:
SYSTEM_NAME = "AI-diff"
BRANCH_NAME = "baseline"

REPORT_FILENAME = "ai_diff_report_b.html"
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL") or os.environ.get("CLAUDE_MODEL", "claude-sonnet-4-5")

system = platform.get_system(SYSTEM_NAME)
branch = system.get_branch(BRANCH_NAME)
print(f"System: {system.name} ({system.id})")
print(f"Branch: {branch.name!r}  snapshot={branch.snapshot_id}")


## 3 · Resolve report and revisions

Look for **`REPORT_FILENAME`** on the branch HEAD. If it exists, read its revision relationships to recover the two models from the last run, then use `get_resource_at_revision` + `is_latest` to see whether either resource has a newer revision.

- **No changes** — print that and leave `PROCEED=False`
- **At least one newer revision** — use those latest revisions and continue
- **No report / no linked pair** — list models on the branch; paste two revision ids in §4


In [ ]:
revisions = branch.list_revisions()


def _resource_type(rev) -> str:
    rt = getattr(rev, "resource_type", None)
    return str(getattr(rt, "value", rt) or "").casefold()


def _is_report(rev) -> bool:
    label = rev.name
    return label == REPORT_FILENAME or (rev.name or "") == REPORT_FILENAME


models_on_branch = [
    rev for rev in revisions
    if rev.resource_id and _resource_type(rev) in {"model", ""}
]
typed = [r for r in models_on_branch if _resource_type(r) == "model"]
if typed:
    models_on_branch = typed

report_on_branch = next((r for r in models_on_branch if _is_report(r)), None)
candidates = [r for r in models_on_branch if not _is_report(r)]

# Filled by auto-resolve or by §4. PROCEED gates later cells.
MODEL_A_REVISION_ID = None
MODEL_B_REVISION_ID = None
EXISTING_REPORT_RESOURCE_ID = None
PROCEED = False


def _list_candidates(reason: str) -> None:
    print(reason)
    print("Set MODEL_A_REVISION_ID and MODEL_B_REVISION_ID from this list:\n")
    for rev in candidates:
        print(f"  {rev.name}")
        print(f"    revision_id: {rev.revision_id}")
        print(f"    resource_id: {rev.resource_id}")


if report_on_branch is None:
    _list_candidates(f"No report named {REPORT_FILENAME!r} on branch {BRANCH_NAME!r}.")
else:
    EXISTING_REPORT_RESOURCE_ID = report_on_branch.resource_id
    report_revision_id = report_on_branch.revision_id
    print(
        f"Found report {_rev_label(report_on_branch)!r} "
        f"revision={report_revision_id}  resource={EXISTING_REPORT_RESOURCE_ID}"
    )

    v3 = platform.v3
    rel_page = v3.list_revision_relationships(revision_id=report_revision_id, size=50)
    rel_items = list(rel_page.items or [])

    # Documents that produced this report (left side of produces).
    linked = []
    for rel in rel_items:
        left = rel.left_revision
        right = rel.right_revision
        if (
            left
            and right
            and right.file_revision_id == report_revision_id
            and left.file_revision_id != report_revision_id
        ):
            linked.append(rel)

    linked.sort(key=lambda r: (r.created is not None, r.created))

    if len(linked) != 2:
        _list_candidates(
            f"Expected 2 linked source revisions on the report; found {len(linked)}."
        )
    else:
        prior_ids = [rel.left_revision.file_revision_id for rel in linked]
        latest_ids = []
        changed = False
        print("Last report compared:")
        for i, prior_id in enumerate(prior_ids, start=1):
            doc = platform.get_resource_at_revision(prior_id)
            latest = doc.latest_revision
            if latest is None:
                print(f"  [{i}] {doc.name}  prior={prior_id}  — no latest revision; stop.")
                latest_ids = []
                changed = False
                break
            latest_ids.append(latest.id)
            is_new = not doc.is_latest
            changed = changed or is_new
            mark = "NEWER" if is_new else "unchanged"
            print(
                f"  [{i}] {doc.name}  prior={doc.revision_id}  latest={latest.id}  ({mark})"
            )

        if len(latest_ids) != 2:
            print("Stopped — could not resolve latest revisions for both sources.")
        elif not changed:
            print(
                "No new revisions on either compared model since the last report — nothing to re-run."
            )
        else:
            MODEL_A_REVISION_ID, MODEL_B_REVISION_ID = latest_ids
            PROCEED = True
            print("\nNewer content found — will compare:")
            print(f"  A revision_id: {MODEL_A_REVISION_ID}")
            print(f"  B revision_id: {MODEL_B_REVISION_ID}")


## 4 · Select models

When §3 did not auto-select, paste two **revision** ids below. **Document A** is the baseline; **document B** is compared against it. If §3 already set the ids, re-run this cell to load them.


In [ ]:
# Paste revision ids only when §3 asked for a manual choice:
MODEL_A_REVISION_ID = "d87b740f-f38c-4bf4-b3d6-f012cc53f67b"
MODEL_B_REVISION_ID = "32e6a244-fa57-4851-908e-d2cd6ea39864"

rev_a = rev_b = None

if MODEL_A_REVISION_ID and MODEL_B_REVISION_ID:
    rev_a = platform.get_revision(MODEL_A_REVISION_ID)
    rev_b = platform.get_revision(MODEL_B_REVISION_ID)
    PROCEED = True
    print(f"A: {rev_a.name} rev:{rev_a.id} of:{rev_a.resource.id}")
    print(f"B: {rev_b.name} rev:{rev_b.id} of:{rev_b.resource.id}")
else:
    print(
        "Stopped. Paste MODEL_A_REVISION_ID and MODEL_B_REVISION_ID above and re-run, "
        "or there is nothing to compare."
    )


## 5 · Prompts

The **system prompt** sets Istari Digital traceability rules (cite artifact UUIDs; do not invent facts). The **user prompt** is the focus for this run — for example what to emphasize when comparing the two documents.

The Messages API is asked for **JSON** (`matches` / `conflicts` / `missing` / `recommendation`). HTML is built locally from the template in §7.


In [ ]:
# Standing instructions (aligned with smart-diff system_prompt.txt).
SYSTEM_PROMPT = """You are a technical document analyst working inside the Istari Digital Platform.

The documents you are comparing are artifacts stored in Istari Digital. Each artifact has a unique artifact ID and revision that serves as its source of truth. Every finding you produce must cite the artifact ID of the document it came from so that all results are fully traceable back to their Istari Digital source.

Only use information explicitly stated in the provided documents. Do not infer, assume, or introduce anything not present in the text.
"""

# Per-run focus — edit this for your documents (smart-diff --prompt).
USER_PROMPT = f"Compare the 2 models: {MODEL_A_REVISION_ID} and {MODEL_B_REVISION_ID}"

if not PROCEED:
    print("Skipped prompts — nothing to compare.")
else:
    print("System prompt:", SYSTEM_PROMPT[:90].replace("\n", " "), "…")
    print("User prompt:", USER_PROMPT)


## 6 · Read documents and invoke Anthropic

Download each revision and extract plain text the same way as smart-diff (`pdfplumber` / `openpyxl` / `python-docx`, or UTF-8 with `errors="replace"` for other types). Ask Claude for **JSON only** in the smart-diff schema.


In [ ]:
import json
import tempfile
import textwrap
from datetime import datetime

import anthropic

if not PROCEED:
    print("Skipped Anthropic call — nothing to compare.")
else:

    def _extract_text(path: Path) -> str:
        """Plain text from PDF / XLSX / DOCX / other — same approach as smart-diff."""
        ext = path.suffix.lower()
        if ext == ".pdf":
            import pdfplumber

            return "\n".join(pg.extract_text() or "" for pg in pdfplumber.open(path).pages)
        if ext == ".xlsx":
            import openpyxl

            wb = openpyxl.load_workbook(path, data_only=True)
            return "\n".join(
                "  |  ".join(str(c) for c in row if c is not None)
                for ws in wb.worksheets
                for row in ws.iter_rows(values_only=True)
            )
        if ext == ".docx":
            from docx import Document

            return "\n".join(para.text for para in Document(path).paragraphs if para.text.strip())
        return path.read_text(errors="replace")

    def _read_revision_text(rev) -> str:
        raw = platform.client.read_contents(token=rev.content_token)
        if isinstance(raw, str):
            raw = raw.encode("utf-8", errors="replace")
        elif not isinstance(raw, (bytes, bytearray)):
            raw = bytes(raw)

        name = rev.name or "document.bin"
        suffix = Path(name).suffix or ".bin"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
            tmp.write(raw)
            tmp_path = Path(tmp.name)
        try:
            return _extract_text(tmp_path)
        finally:
            tmp_path.unlink(missing_ok=True)

    def _strip_json_fences(text: str) -> str:
        text = text.strip()
        fenced = re.match(r"^```(?:json)?\s*([\s\S]*?)```\s*$", text, re.IGNORECASE)
        return fenced.group(1).strip() if fenced else text

    filename_a = rev_a.name or rev_a.display_name or "document_a"
    filename_b = rev_b.name or rev_b.display_name or "document_b"
    uuid_a, rev_id_a = rev_a.resource.id, rev_a.id
    uuid_b, rev_id_b = rev_b.resource.id, rev_b.id

    text_a = _read_revision_text(rev_a)
    text_b = _read_revision_text(rev_b)
    print(f"Document A ({filename_a}): {len(text_a)} characters")
    print(f"Document B ({filename_b}): {len(text_b)} characters")

    # Same message shape as smart-diff: JSON schema + user focus + labeled documents.
    # Keep document bodies outside dedent so their whitespace is preserved.
    user_content = textwrap.dedent(f"""\
        Return ONLY valid JSON in this exact format:
        {{"matches":["..."],"conflicts":[{{"item":"","value1":"","value2":""}}],"missing":[{{"item":"","missing_from":"","detail":""}}],"recommendation":"..."}}

        User focus: {USER_PROMPT}
        """)
    user_content += (
        f"\n--- Document 1 | {filename_a} | UUID: {uuid_a} | Revision: {rev_id_a} ---\n"
        f"{text_a}"
        f"\n\n--- Document 2 | {filename_b} | UUID: {uuid_b} | Revision: {rev_id_b} ---\n"
        f"{text_b}"
    )

    llm = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    message = llm.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=4096,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_content}],
    )

    raw = "".join(
        block.text for block in message.content if getattr(block, "type", None) == "text"
    )
    diff = json.loads(_strip_json_fences(raw))
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(f"Model: {ANTHROPIC_MODEL}")
    print(f"Stop reason: {message.stop_reason}")
    print(
        f"Findings — matches: {len(diff.get('matches', []))}, "
        f"conflicts: {len(diff.get('conflicts', []))}, "
        f"missing: {len(diff.get('missing', []))}"
    )


## 7 · Render HTML report

Fill [`report_template.html`](report_template.html) with the JSON findings and a **source of truth** block. Write the HTML report and `_prompt.txt` audit file under a temp directory (not next to this notebook).


In [ ]:
if not PROCEED:
    print('Skipped HTML render — nothing to compare.')
else:
    import tempfile
    from string import Template

    matches_html = "".join(f"<li>{m}</li>" for m in diff["matches"])
    conflicts_html = "".join(
        f'<tr style="border-bottom:1px solid #ddd">'
        f'<td style="padding:8px">{c["item"]}</td>'
        f'<td style="padding:8px">{c["value1"]}</td>'
        f'<td style="padding:8px">{c["value2"]}</td></tr>'
        for c in diff["conflicts"]
    )
    missing_html = "".join(
        f'<li><b>{m["missing_from"]}</b> did not specify {m["item"]}. {m.get("detail", "")}</li>'
        for m in diff["missing"]
    )

    html_report = Template((NOTEBOOK_DIR / "report_template.html").read_text(encoding="utf-8")).substitute(
        filename1=filename_a,
        filename2=filename_b,
        uuid1=uuid_a,
        rev1=rev_id_a,
        uuid2=uuid_b,
        rev2=rev_id_b,
        provider="claude",
        model=ANTHROPIC_MODEL,
        timestamp=timestamp,
        matches_html=matches_html,
        conflicts_html=conflicts_html,
        missing_html=missing_html,
        recommendation=diff["recommendation"],
    )

    REPORT_DIR = Path(tempfile.mkdtemp(prefix="ai-validation-"))
    report_path = REPORT_DIR / REPORT_FILENAME
    report_path.write_text(html_report, encoding="utf-8")

    prompt_audit = REPORT_DIR / (report_path.stem + "_prompt.txt")
    prompt_audit.write_text(
        f"PROMPT\n{'=' * 40}\n{USER_PROMPT}\n\nPROVIDER: claude\nMODEL: {ANTHROPIC_MODEL}\n",
        encoding="utf-8",
    )

    print(f"Wrote {report_path.resolve()}")
    print(f"Wrote {prompt_audit.resolve()}")
    display(HTML(html_report))


## 8 · Track the report on the branch

- **First run** (no report on the branch) — upload a new model, track it, advance the branch HEAD
- **Later runs** — upload a new revision of the existing report model, then snapshot/advance so the branch picks up LATEST

> Pause in the Istari Digital web app: open the system → confirm the report appears on the branch.


In [ ]:
if not PROCEED:
    print('Skipped report upload — nothing to compare.')
else:
    branch = system.get_branch(BRANCH_NAME)
    version_label = timestamp.replace(" ", "T").replace(":", "")

    if EXISTING_REPORT_RESOURCE_ID:
        updated = platform.client.update_model(
            EXISTING_REPORT_RESOURCE_ID,
            report_path,
            version_name=version_label
        )
        report_doc = platform.get_model(updated.id)
        # Same configuration; new snapshot resolves LATEST to this revision.
        branch.advance_to(branch.configuration)
        print(f"Updated report model {report_doc.id} → revision {report_doc.revision_id}")
    else:
        report_doc = platform.upload_model(
            report_path,
            external_id=f"ai-diff-report-{version_label}"
        )
        new_cfg = branch.add_resource(report_doc).save()
        branch.advance_to(new_cfg)
        print(f"Created report model {report_doc.id}  revision={report_doc.revision_id}")
        print(f"Tracked on configuration: {new_cfg.name} ({new_cfg.id})")

    print(f"Advanced branch {BRANCH_NAME!r} → snapshot {branch.snapshot_id}")
    print(f"System: {system.name} ({system.id})")


## 9 · Link resources with a revision relationship

Revision relationships connect two **file revisions** (not resource ids directly). Use `rev_a` / `rev_b` from §4 (`rev.id` and `rev.resource.id`), then create a `produces` edge with `platform.v3` — same pattern as [workflow log Scenario A](../workflow-logs/workflow_log_scenario_a.ipynb).

This sample links each compared document to the HTML report (`left` produces `right`).


In [ ]:
if not PROCEED:
    print('Skipped relationships — nothing to compare.')
else:
    from istari_digital_client.v3.models.new_revision_relationship_dto import NewRevisionRelationshipDto

    v3 = platform.v3
    rel_types = list(v3.list_revision_relationship_types(size=50).items or [])
    produces = next((t for t in rel_types if t.name == "produces"), None)
    if produces is None:
        raise RuntimeError(
            "No 'produces' relationship type on this registry. "
            f"Available: {[t.name for t in rel_types] or '(none)'}"
        )

    pairs = [
        (rev_a.resource.id, rev_a.id, "document A"),
        (rev_b.resource.id, rev_b.id, "document B"),
    ]

    print(f"Relationship type: {produces.name} ({produces.id})")
    print(f"Report resource={report_doc.id}  revision={report_doc.revision_id}\n")

    for resource_id, revision_id, label in pairs:
        rel = v3.create_revision_relationship(
            new_revision_relationship_dto=NewRevisionRelationshipDto(
                relationship_type_id=produces.id,
                left_revision_id=revision_id,
                right_revision_id=report_doc.revision_id,
            )
        )
        print(f"Linked {label}")
        print(f"  left  resource={resource_id}  revision={revision_id}")
        print(f"  right resource={report_doc.id}  revision={report_doc.revision_id}")
        print(f"  relationship id: {getattr(rel, 'id', rel)}")


## Learn more

- [Key Concepts](https://docs.istaridigital.com/intro/key-concepts)
- [Python Client — Quick Start](https://docs.istaridigital.com/developers/SDK/setup)
- [Anthropic Messages API](https://docs.anthropic.com/en/api/messages)
- [Download system resources](../resources/download-system-resources.ipynb) — export every revision on a branch
- [External workflow logs — Scenario A](../workflow-logs/workflow_log_scenario_a.ipynb) — `produces` revision relationships
